# T4 노트북에서 Isaac Sim 실행 점검

이미 실행했다면 **첫 코드 셀만** 실행해 현재 폴더의 `pickloop-isaac.log`에서 물리 측정값과 첫 오류를 확인하세요. 파일은 그대로 유지됩니다. 다른 런타임을 열었다면 기존 로그 파일을 먼저 옮겨야 합니다.

환경 점검 셀은 할당된 GPU, 드라이버, 메모리와 Vulkan 상태를 조회합니다. 설치나 API 키가 필요하지 않습니다. Colab에서는 GPU 런타임을, Kaggle에서는 GPU 가속기를 선택하세요.

마지막 셀은 선택 실험입니다. 한 장의 GPU로 **Isaac Sim 5.1.0**을 시작해 바닥 위로 큐브를 떨어뜨리고 180회 물리 스텝 후 높이를 검사합니다. 이전 릴리스 5.1.0은 NVIDIA의 유지보수가 종료됐으며, 이 시험은 해당 버전으로 고정했습니다. T4 두 장은 GPU당 메모리를 각각 사용하며, T4는 이 버전의 공식 최소 GPU인 RTX 4080보다 낮은 사양입니다. 이 시험의 성공 여부는 현재 노트북 세션에서 확인합니다. 인터넷과 충분한 여유 디스크가 필요합니다(NVIDIA 최소 저장 공간 50 GB). 실행 중 노트북 GPU 할당량이 소모될 수 있습니다.

[5.1 시스템 요구사항](https://docs.isaacsim.omniverse.nvidia.com/5.1.0/installation/requirements.html) · [NVIDIA Python 설치](https://docs.isaacsim.omniverse.nvidia.com/5.1.0/installation/install_python.html) · [NVIDIA 이용 약관](https://docs.isaacsim.omniverse.nvidia.com/5.1.0/common/NVIDIA_Omniverse_License_Agreement.html)

In [ ]:
# 기존 로그 복구 · 설치 및 GPU 실행 없이 읽기
import json
import math
from pathlib import Path

def read_physics_log(log):
    evidence = {}
    count = 0
    with log.open(errors="replace") as stream:
        for raw in stream:
            line = raw.rstrip()
            if line.startswith("PICKLOOP_STAGE "):
                evidence["last_stage"] = line.removeprefix("PICKLOOP_STAGE ")
            elif line.startswith("PICKLOOP_PHYSICS_RESULT "):
                count += 1
                try:
                    sample = json.loads(line.removeprefix("PICKLOOP_PHYSICS_RESULT "))
                    start, end, speed = (sample[key] for key in ("start_z_m", "final_z_m", "final_speed_m_s"))
                    if not all(type(v) in (int, float) and math.isfinite(v) for v in (start, end, speed)):
                        raise ValueError("nonfinite or invalid measurement")
                    passed = .95 <= start <= 1.05 and .03 <= end <= .07 and speed < .03
                    if sample["physics_steps"] != 180 or speed < 0 or sample["passed"] is not passed:
                        raise ValueError("inconsistent measurement")
                    evidence["physics"] = sample
                except (KeyError, TypeError, ValueError, OverflowError):
                    evidence["measurement_error"] = "invalid_measurement"
            if "Fatal Python error:" in line and "fatal_error" not in evidence:
                evidence["fatal_error"] = line[:500]
            if ("[Error]" in line or line.startswith("Traceback (most recent call last)")) and "first_logged_error" not in evidence:
                evidence["first_logged_error"] = line[:500]
    evidence["measurement_lines"] = count
    if count != 1:
        evidence.pop("physics", None)
    return evidence

existing_log = Path("pickloop-isaac.log")
if existing_log.is_file():
    print(json.dumps(read_physics_log(existing_log), ensure_ascii=False, indent=2))
else:
    print("현재 폴더에 pickloop-isaac.log가 없습니다.")

In [ ]:
import ctypes.util
import json
import os
import platform
import shutil
import subprocess
from pathlib import Path

def capture(command):
    if not shutil.which(command[0]):
        return {"status": "tool_missing"}
    try:
        p = subprocess.run(command, capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return {"status": "timeout"}
    return {"exit_code": p.returncode, "output": (p.stdout + p.stderr).strip()[:8000]}

work = Path.cwd()
report = {
    "isaac_version_requested": "5.1.0",
    "system": platform.platform(),
    "kernel_python": platform.python_version(),
    "available_disk_GiB": round(shutil.disk_usage(work).free / 2**30, 1),
    "system_ram_GiB": round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 1),
    "gpu": capture(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"]),
    "vulkan_loader": ctypes.util.find_library("vulkan"),
    "vulkan_summary": capture(["vulkaninfo", "--summary"]),
    "status": "preflight_only",
}
result_path = work / "pickloop-isaac-result.json"

def save():
    result_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")

if not result_path.exists():
    save()
print(result_path)
print(json.dumps(report, ensure_ascii=False, indent=2))

## 한 번의 물리 시험

실제로 설치하고 실행하려면 아래 셀의 `RUN_PHYSICS`를 `True`로 바꾸세요. 위에 연결한 NVIDIA 약관을 읽고 동의한 경우에만 `ACCEPT_NVIDIA_EULA`를 `True`로 바꾸세요. 노트북에 인터넷이 허용되어 있어야 합니다. 별도 환경에 Python 3.11과 패키지를 설치합니다. 결과 JSON과 설치·실행 로그는 현재 폴더에 남습니다. `physics_passed`는 이 버전에서 단일 큐브의 낙하·정지가 관측됐다는 뜻입니다. PickLoop 피킹이나 카메라 렌더링은 별도 실험입니다.

재실행 시 직전 로그와 결과 JSON은 현재 폴더의 `pickloop-isaac-prior-*`에 보관합니다. `physics.passed`는 측정 판정이고, `physics_exit_code`는 프로세스 종료 코드입니다. 측정이 끝난 뒤 비정상 종료하면 측정값을 보존하고 전체 상태는 `run_incomplete`로 기록합니다.

In [ ]:
RUN_PHYSICS = False
ACCEPT_NVIDIA_EULA = False

import tempfile
import venv

def run_smoke():
    if not RUN_PHYSICS or not ACCEPT_NVIDIA_EULA:
        print("설치·물리 시험은 선택 실행입니다. 현재 결과:", report["status"])
        return
    if report["gpu"].get("exit_code") != 0:
        print("GPU 런타임에서 첫 번째 셀을 다시 실행하세요.")
        return

    root = work / ".pickloop-isaac-5.1"
    root.mkdir(exist_ok=True)
    log = work / "pickloop-isaac.log"
    env = {**os.environ, "OMNI_KIT_ACCEPT_EULA": "YES", "PIP_DISABLE_PIP_VERSION_CHECK": "1"}
    program = '''
import json
import numpy as np
from isaacsim import SimulationApp
print("PICKLOOP_STAGE starting_app", flush=True)
app = SimulationApp({"headless": True, "active_gpu": 0, "physics_gpu": 0, "multi_gpu": False})
try:
    print("PICKLOOP_STAGE building_world", flush=True)
    from isaacsim.core.api import World
    from isaacsim.core.api.objects import DynamicCuboid, FixedCuboid
    world = World(physics_dt=1/60, rendering_dt=1/60, stage_units_in_meters=1.0)
    world.scene.add(FixedCuboid(prim_path="/World/Bench", name="bench", size=1.0,
                                scale=np.array([3.0, 3.0, 0.1]), position=np.array([0., 0., -.05])))
    cube = world.scene.add(DynamicCuboid(prim_path="/World/Box", name="box", size=.1, mass=.1,
                                         position=np.array([0., 0., 1.])))
    print("PICKLOOP_STAGE resetting_world", flush=True)
    world.reset()
    start = float(cube.get_world_pose()[0][2])
    print("PICKLOOP_STAGE stepping_world", flush=True)
    for _ in range(180):
        world.step(render=False)
    end = float(cube.get_world_pose()[0][2])
    speed = float(np.linalg.norm(cube.get_linear_velocity()))
    passed = .95 <= start <= 1.05 and .03 <= end <= .07 and speed < .03
    print("PICKLOOP_PHYSICS_RESULT " + json.dumps(dict(start_z_m=start, final_z_m=end,
          final_speed_m_s=speed, physics_steps=180, passed=passed), allow_nan=False), flush=True)
except Exception:
    import traceback
    traceback.print_exc()
    raise
finally:
    print("PICKLOOP_STAGE closing_app", flush=True)
    app.close()
    print("PICKLOOP_STAGE app_close_returned", flush=True)
'''
    if log.exists():
        previous = Path(tempfile.mkdtemp(prefix="pickloop-isaac-prior-", dir=work))
        shutil.copy2(log, previous / log.name)
        if result_path.exists():
            shutil.copy2(result_path, previous / result_path.name)
        print("직전 실행 보관:", previous)
    for key in ("physics", "error", "physics_exit_code", "measurement_lines", "last_stage",
                "fatal_error", "first_logged_error", "measurement_error"):
        report.pop(key, None)
    report["status"] = "installing"
    save()
    try:
        with log.open("w") as out:
            def run(args, timeout=1200):
                subprocess.run([str(a) for a in args], stdout=out, stderr=subprocess.STDOUT,
                               check=True, timeout=timeout, env=env)
            # Install uv in a private helper environment when it is absent from the notebook.
            uv = shutil.which("uv")
            if uv is None:
                helper = root / "tools"
                venv.create(helper, with_pip=True)
                run([helper/"bin/python", "-m", "pip", "install", "uv"])
                uv = helper / "bin/uv"
            target = root / "runtime"
            python = target / "bin/python"
            if not python.exists():
                run([uv, "venv", "--seed", "--python", "3.11", target])
            run([python, "-m", "pip", "install", "--no-cache-dir", "--extra-index-url", "https://pypi.nvidia.com",
                 "isaacsim[all,extscache]==5.1.0"])
            report["status"] = "starting_physics"
            save()
            try:
                child = subprocess.run([str(python), "-u", "-c", program], stdout=out,
                                       stderr=subprocess.STDOUT, env=env, timeout=600, check=False)
                report["physics_exit_code"] = child.returncode
            except subprocess.TimeoutExpired:
                report["physics_exit_code"] = None
                report["error"] = "physics process timed out after 600 s"
        report.update(read_physics_log(log))
        if report["physics_exit_code"] != 0 or "physics" not in report:
            report["status"] = "run_incomplete"
            if "error" not in report:
                report["error"] = ("physics process exit " + str(report["physics_exit_code"])
                                   if report["physics_exit_code"] else "missing or invalid physics result")
        else:
            report["status"] = "physics_passed" if report["physics"]["passed"] else "physics_failed"
    except (subprocess.SubprocessError, OSError, RuntimeError, ValueError) as exc:
        report["error"] = f"{type(exc).__name__} at {report['status']}"
        report["status"] = "run_incomplete"
        if log.exists():
            from collections import deque
            with log.open(errors="replace") as f:
                print("".join(deque(f, maxlen=12)))
    finally:
        save()
    print(json.dumps(report, ensure_ascii=False, indent=2))

run_smoke()